# How I built the Tri-Modal Insurance Claim Gating Benchmark

Each sample contains a CarDD vehicle photo, a categorical claim description, and 11 numerical features from the original insurance claim. The claim and photo come from independent sources. The **pairing and routing label are synthetic**. The numerical values are copied from the matched claim.

This notebook explains the original construction and audits the exported dataset. It replays the numerical join when the original CSV and build provenance are available. It uses the published assignments for the audit.

## 1. Load the benchmark
Run from the folder containing this notebook and the dataset's `data/` folder.

In [ ]:
from collections import Counter
from decimal import Decimal
from pathlib import Path
import csv
import hashlib
import json

root = Path.cwd()
data = root / "data"
name = "multimodal_claims_benchmark"
rows = [json.loads(line) for line in (data / f"{name}.jsonl").read_text().splitlines()]
with (data / f"{name}.csv").open(newline="", encoding="utf-8") as stream:
    csv_rows = list(csv.DictReader(stream))
print(f"{len(rows):,} samples and {len(rows[0])} fields")

## 2. Understand the two sources

- **Auto Insurance Claims Data** supplied 1,000 claim records. The text and numerical features for a sample come from the same record.
- **CarDD** supplied 4,000 photos and 8,740 annotations across six damage categories. To estimate visible damage extent, the build used the union of the annotated bounding boxes as a fraction of image area. Combining overlapping boxes this way avoids counting the overlap twice.

[Claims source](https://www.kaggle.com/datasets/buntyshah/auto-insurance-claims-data) · [CarDD](https://cardd-ustc.github.io/)

In [ ]:
def visual_extent(categories, box_coverage, box_count):
    surface_damage = {"dent", "scratch"}
    if set(categories) <= surface_damage and box_coverage <= 0.12:
        return "localized"
    if box_coverage >= 0.24 and (box_count >= 2 or set(categories) - surface_damage):
        return "extensive"
    return "moderate"

print(visual_extent(["scratch"], 0.04, 1))
print(visual_extent(["dent", "lamp broken"], 0.33, 2))

## 3. Assign the routing labels

| Decision | How the original build paired claims and images |
| --- | --- |
| Auto-Approve | Used eligible claims with fraud flag N, recorded police-report status, and clear claim fields. It paired trivial claims with localized damage, minor claims with localized or moderate damage, and major claims with extensive damage. |
| Reject_Fraud_Audit | Paired major damage or total loss with a localized photo. A fraud-flagged minor or trivial claim could instead be paired with an extensive photo. |
| Escalate_Human | Used claims with missing or ambiguous evidence, such as an unknown police-report status or theft. It applied blur, low lighting, or a heavy crop to the photo. |

An integer optimizer allocated claims and images across the eligible combinations. It first maximized the number of unique claims, then minimized deviations from the class targets. The publication used seed 42. Eligibility capped auto-approvals at 271. Source photos were screened for exact or perceptual duplicates before assignment.

In [ ]:
for label, count in Counter(row["decision_label"] for row in rows).items():
    print(f"{label:22} {count:3}  ({count / len(rows):.1%})")

## 4. Split the paired records

The original build stratified the assignments into 70% train, 15% validation, and 15% test. Each source image and claim was used once. The tri-modal update reused the existing splits.

In [ ]:
split_classes = Counter((row["split"], row["decision_label"]) for row in rows)
labels = ("Auto-Approve", "Reject_Fraud_Audit", "Escalate_Human")
print("Split      Approve   Audit   Escalate   Total")
for split in ("train", "val", "test"):
    counts = [split_classes[split, label] for label in labels]
    print(f"{split:10} {counts[0]:7} {counts[1]:7} {counts[2]:10} {sum(counts):7}")

## 5. Add the numerical modality

The tri-modal update joined each existing `sample_id` to its original claim row using the build provenance. It kept the original fields and added a separate `claim_description` with incident type, collision type, severity, vehicle make and model, and police-report status. Use that field as the model text input. The legacy summary includes amounts.

For the optional source replay, set `claims_path` and `provenance_path` to copies of the original files. The two files are separate from the distributed benchmark.

In [ ]:
numeric = (
    "total_claim_amount", "vehicle_claim", "injury_claim", "property_claim",
    "policy_deductable", "policy_annual_premium", "age", "months_as_customer",
    "auto_year", "bodily_injuries", "witnesses",
)

claims_path = root / "insurance_claims.csv"
provenance_path = root / "benchmark_provenance.jsonl"

if claims_path.is_file() and provenance_path.is_file():
    with claims_path.open(encoding="utf-8-sig", newline="") as stream:
        original_claims = list(csv.DictReader(stream))
    provenance = {
        item["sample_id"]: item
        for item in map(json.loads, provenance_path.read_text().splitlines())
    }
    assert len({provenance[row["sample_id"]]["claim_data_row"] for row in rows}) == len(rows)
    for row in rows:
        ref = provenance[row["sample_id"]]
        claim = original_claims[ref["claim_data_row"] - 1]
        for field in numeric:
            assert Decimal(str(row[field])) == Decimal(claim[field])
        assert hashlib.sha256((root / row["image_path"]).read_bytes()).hexdigest() == ref["image_sha256"]
    print(f"Verified {len(rows) * len(numeric):,} original source values and {len(rows):,} image hashes.")
else:
    print("Source replay requires the original claims CSV and build provenance.")

## 6. Check the distributed files

In [ ]:
assert len(rows) == len(csv_rows) == 1000
assert len({r["sample_id"] for r in rows}) == 1000
for row, csv_row in zip(rows, csv_rows, strict=True):
    assert row.keys() == csv_row.keys()
    assert all(row[field] is not None and row[field] != "" for field in row)
    for field in numeric:
        expected_type = float if field == "policy_annual_premium" else int
        assert type(row[field]) is expected_type
        assert Decimal(csv_row[field]) == Decimal(str(row[field]))
    assert all(row[field] == csv_row[field] for field in row.keys() - set(numeric))
paths = [root / row["image_path"] for row in rows]
assert len(set(paths)) == 1000
assert set(paths) == set((data / "benchmark_images").glob("*.jpg"))
print("CSV and JSONL agree. All 1,000 image paths resolve. No missing numerical values.")

## 7. Show a single model input
The decision label is the answer for scoring. The pairing reason explains how it was assigned and should stay out of the model prompt.

In [ ]:
sample = rows[0]
for field in ("sample_id", "image_path", "claim_description"):
    print(sample[field])
print({field: sample[field] for field in numeric})
print("Expected decision:", sample["decision_label"])

## What the benchmark measures

The labels test routing under constructed image–claim conditions. The visible-damage proxy depends on camera framing. When presenting results, compare text-only, numerical-only, and combined inputs so readers can see what the photo contributes. Some test descriptions also appear in training, since several claims share the same categorical details.